In [1]:
import pandas as pd
import numpy as np
from collections import Counter
import ast
import matplotlib.pyplot as plt
import torch
import math
import seaborn as sns
from pathlib import Path
import os

### 1. Folder and file definitions
Based on the threshold analysis in threshold_analysis.ipynb, we choose the threshold of ... % and use this throughout the entire analysis in this file.

In [2]:
RESULT_DIR = Path("../results/oct2410_GCN_train_50w_wan-w5_nf-emb_ew-wan_sp-random_L2_H64_sum_lr0.001_seed2029")
motif_file = RESULT_DIR / "noob_motcom040.csv" 
results_file = RESULT_DIR / "explanations_none_object.pth"

In [3]:
df_motifs = pd.read_csv(motif_file)
df_motifs

,motif,human_present,human_selected,human_rate,machine_present,machine_selected,machine_rate,difference
0,"((263, 33), (263, 183))",1,1,1.0,2,0,0.0,-1.0
1,"((263, 33), (263, 269))",1,1,1.0,1,0,0.0,-1.0
2,"((263, 113), (263, 114))",2,0,0.0,1,1,1.0,1.0
3,"((266, 113), (266, 164))",1,0,0.0,1,1,1.0,1.0
4,"((266, 197), (266, 198))",3,0,0.0,1,1,1.0,1.0
...,...,...,...,...,...,...,...,...
19521,"((273, 84), (273, 224))",1,1,1.0,0,0,NaN,NaN
19522,"((273, 105), (273, 197))",1,1,1.0,0,0,NaN,NaN
19523,"((273, 108), (273, 197))",1,1,1.0,0,0,NaN,NaN
19524,"((273, 163), (273, 198))",1,1,1.0,0,0,NaN,NaN


### 2. Summary motifs

In [6]:
human_selected = df_motifs["human_selected"] > 0
machine_selected = df_motifs["machine_selected"] > 0

human_only = (human_selected & ~machine_selected).sum()
machine_only = (~human_selected & machine_selected).sum()
shared = (human_selected & machine_selected).sum()

total = (human_selected | machine_selected).sum()

print(f"Total unique selected motifs: {total}")
print(f"Human only: {human_only}")
print(f"Machine only: {machine_only}")
print(f"Shared: {shared}")

Total unique selected motifs: 19526
Human only: 14932
Machine only: 4319
Shared: 275


In [7]:
h = df_motifs[
    (df_motifs["human_selected"] > 0) &
    (df_motifs["machine_selected"] == 0)
]

m = df_motifs[
    (df_motifs["human_selected"] == 0) &
    (df_motifs["machine_selected"] > 0)
]

print("Human-only motifs selected once (uniqe motifs within the human class):",
      (h["human_selected"] == 1).sum())

print("Amount of non-unique motifs within the human class: ",
      human_only - (h["human_selected"] == 1).sum())

print("Machine-only motifs selected once (unique motifs within the machine class):",
      (m["machine_selected"] == 1).sum())

print("Amount of non-unique motifs within the machine class: ",
      machine_only - (m["machine_selected"] == 1).sum())

Human-only motifs selected once (uniqe motifs within the human class): 13234
Amount of non-unique motifs within the human class:  1698
Machine-only motifs selected once (unique motifs within the machine class): 3695
Amount of non-unique motifs within the machine class:  624


In [8]:
results = torch.load(
    results_file,
    map_location="cpu",
    weights_only=False
)

In [9]:
df_explanations = pd.DataFrame([
    {
        "row": result["row"],
        "label": result["label"],
        "pred": result["pred"]
    }
    for result in results
])

# Count unique graphs per class
graph_counts = (
    df_explanations
    .groupby("label")["row"]
    .nunique()
)

print("Number of graphs per class:")
print(graph_counts)

print("Total unique graphs:", df_explanations["row"].nunique())

Number of graphs per class:
label
0    180
1    180
Name: row, dtype: int64
Total unique graphs: 360


In [10]:
df_correct = df_explanations[
    df_explanations["label"] == df_explanations["pred"]
]

n_human_graphs = df_correct[
    df_correct["label"] == 0
]["row"].nunique()

n_machine_graphs = df_correct[
    df_correct["label"] == 1
]["row"].nunique()

print("Correctly classified human graphs:", n_human_graphs)
print("Correctly classified machine graphs:", n_machine_graphs)

Correctly classified human graphs: 163
Correctly classified machine graphs: 172
